# Six Layers That Beat The Top Public Agents

**TL;DR**: Stack six public micro-optimizations from three different authors on Tschinkel's v13 base.
560 wins, 54 losses, 36 ties across 14 opponents — **88.9% Elo rate**.

Near-parity with ahmed v56, the strongest public agent on the leaderboard.

The six layers:
- **CL** — Crop Longevity (extend field life when profitable)
- **Price Guard** — avoid selling below cost in volatile markets
- **Race Horizon** — look-ahead planting based on competitor timing
- **EXP402** — Late Seed Cap (stop buying seeds when ROI window closes)
- **EXP410** — Fertilizer Guard (skip fertilizer when payoff is negative)
- **IG** — Queue Compaction (merge redundant queue entries)

No datasets, no internet, no GPU. Everything is embedded below.

---

## How We Found Them: Autoloop Layer Stacking

We ran our automated pipeline (autoloop v5) to scout, extract, and validate layers from top public agents.
Each layer was isolated via tape-to-tape diffing against the Tschinkel base, tested individually,
then stacked incrementally. Only layers that improved Elo rate in the full gauntlet survived.

### The Six Layers

1. **CL (Crop Longevity)** — From dmitriigluzdov's agent. Extends field usage when the crop
   still has positive expected value, rather than replanting on a fixed schedule.

2. **Price Guard** — From ahmed's v55/v56 line. Refuses to sell harvested crops when the current
   market price is below production cost, waiting for a better price window.

3. **Race Horizon** — From ahmed's v55/v56 line. Adjusts planting timing based on what
   competitors are likely to plant, avoiding head-to-head supply gluts.

4. **EXP402 (Late Seed Cap)** — Experimental layer. Stops purchasing seeds when the remaining
   game turns are too few for the crop to break even.

5. **EXP410 (Fertilizer Guard)** — Experimental layer. Skips fertilizer application when the
   expected yield boost doesn't cover the fertilizer cost.

6. **IG (Queue Compaction)** — From lynnsakurai's agent. Merges consecutive identical queue
   entries into single operations, freeing action slots for higher-value moves.

---

## Validation: 650 Games Across 14 Opponents

| Opponent | W | L | T | Win Rate |
|----------|---|---|---|----------|
| ahmed v56 | 28 | 16 | 6 | 56.0% |
| ahmed v55 | 35 | 10 | 5 | 70.0% |
| lynnsakurai | 42 | 4 | 4 | 84.0% |
| dmitriigluzdov | 44 | 2 | 4 | 88.0% |
| tschinkel v13 | 46 | 1 | 3 | 92.0% |
| (11 others) | 365 | 21 | 14 | 91.3% |
| **Total** | **560** | **54** | **36** | **88.9%** |

The hardest matchup is ahmed v56 — we're near-parity there.
Against everyone else, the stacked layers dominate.

---

## Build the Agent

No datasets, no internet, no GPU. The complete agent is embedded below.

In [ ]:
from pathlib import Path
import os, sys, time

WORKDIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

In [ ]:
import base64, gzip, hashlib

EXPECTED_SHA256 = 'a4a563c66ea84746cdb5b18341e78d97677ba5a8cef67e9a157afdaf1d6207d7'

# pipe-18: Tschinkel v13 base + CL + Price Guard + Race Horizon + EXP402 + EXP410 + IG
# Compressed and base64-encoded. 6,878 lines, 1,022,762 bytes uncompressed.
AGENT_B64 = (
    "..."
)

source_bytes = gzip.decompress(base64.b64decode(AGENT_B64))
actual_sha = hashlib.sha256(source_bytes).hexdigest()
assert actual_sha == EXPECTED_SHA256, f'SHA256 mismatch! {actual_sha} != {EXPECTED_SHA256}'

agent_path = WORKDIR / 'main.py'
agent_path.write_bytes(source_bytes)
print(f'Wrote {len(source_bytes):,} bytes to {agent_path}')
print(f'SHA256 verified: {actual_sha[:16]}...')

In [ ]:
import ast

# Verify the agent parses and has the expected entry point
tree = ast.parse(source_bytes.decode('utf-8', errors='replace'))
functions = [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]
assert 'agent' in functions, f'No agent() function found! Functions: {functions[:10]}'
print(f'Verified: agent() found among {len(functions)} functions')

## Build the Submission Archive

In [ ]:
import gzip as gzip_mod, io, tarfile
from IPython.display import FileLink, display

ARCHIVE = WORKDIR / 'submission.tar.gz'

buffer = io.BytesIO()
with tarfile.open(fileobj=buffer, mode='w', format=tarfile.GNU_FORMAT) as tar:
    info = tarfile.TarInfo('main.py')
    info.size, info.mtime, info.mode = len(source_bytes), 0, 0o644
    tar.addfile(info, io.BytesIO(source_bytes))
ARCHIVE.write_bytes(gzip_mod.compress(buffer.getvalue(), mtime=0))

sz = ARCHIVE.stat().st_size
print(f'Created {ARCHIVE} ({sz:,} bytes / {sz/1024:.0f} KB)')

# Verify
with tarfile.open(ARCHIVE) as tar:
    names = tar.getnames()
    extracted = tar.extractfile('main.py').read()
    assert extracted == source_bytes, 'Content mismatch!'
    print(f'Contents: {names} — verified identical')

print('Ready to submit!')
display(FileLink(str(ARCHIVE)))

---

# Attribution and License

This agent is a derivative work. Full credit to the authors:

### Base agent: Thomas Tschinkel
- Notebook: "KAAG Is All You Need" (v13)
- License: Apache 2.0
- The core game engine, planting logic, and economic model are his work.

### Layer contributor: Dmitrii Gluzdov
- Notebook: "A Smaller Market"
- The **CL (Crop Longevity)** concept — extending field life based on expected value — originated from his agent.

### Layer contributor: Ahmed Berat Ozer
- Notebooks: v55, v56 series
- The **Price Guard** and **Race Horizon** layers were extracted from his agent line,
  which represents the strongest public agent on the leaderboard.

### Layer contributor: lynnsakurai
- Notebook: "Farming Score v2"
- The **IG (Queue Compaction)** layer was extracted from this agent.

### Experimental layers: autoloop pipeline
- **EXP402 (Late Seed Cap)** and **EXP410 (Fertilizer Guard)** were discovered
  through automated ablation experiments on the stacked agent.

All modifications are documented. The derivative is released under Apache 2.0,
consistent with the base license.